## Overview

BikeEase has successfully implemented various AI-powered solutions for demand forecasting, customer review analysis, and image classification. As they continue to grow, they aim to automate certain tasks using Large Language Models (LLMs), particularly in marketing and advertising generation to attract more customers and increase engagement.

To achieve this, BikeEase plans to develop a Generative AI-powered system that can automatically create engaging and persuasive advertisements based on bike specifications, discount offers, and promotional themes. This will enable them to generate high-quality marketing content without manual effort, saving time and ensuring brand consistency

## Project Statement

Develop a Generative AI-powered advertisement generation system using LLMs and LangChain to create compelling promotional content for BikeEase’s rental services

## Steps to Perform

### Task 1: Understand generative AI & LLMs

- Explore how LLMs can be used for automated marketing
- Learn about LangChain and how it helps integrate LLMs into applications

### Task 2: Designing the Ad generation pipeline

- Accept user inputs for bike specifications, discount options, and marketing themes
- Use LLMs (Hugging Face models) to generate creative, engaging ads
- Structure the output to align with BikeEase’s branding and tone

### Task 3: Building the LLM-based Ad generator

- Use LangChain to manage the prompt engineering process
- Integrate a local Hugging Face model to generate text without API dependencies
- Experiment with different prompt techniques to enhance response quality

### Task 4: Evaluation and optimization

- Test the ad variations to ensure quality, persuasiveness, and relevance.
- Implement prompt tuning to fine-tune outputs for different use cases.
- Compare different LLM models to identify the most effective one for marketing

In [1]:
%pip install --quiet --upgrade pip
%pip install  transformers datasets evaluate peft bitsandbytes accelerate langchain scikit-build-core openai
%pip install langchain


Note: you may need to restart the kernel to use updated packages.
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
  Using cached langchain-1.2.13-py3-none-any.whl.metadata (5.8 kB)
  Using cached langgraph-1.1.3-py3-none-any.whl.metadata (7.4 kB)
  Using cached langgraph_prebuilt-1.0.8-py3-none-any.whl.metadata (5.2 kB)
Using cached evaluate-0.4.6-py3-none-any.whl (84 kB)
Using cached peft-0.18.1-py3-none-any.whl (556 kB)
Using cached langchain-1.2.13-py3-none-any.whl (112 kB)
Using cached langgraph-1.1.3-py3-none-any.whl (168 kB)
Using cached langgraph_prebuilt-1.0.8-py3-none-any.whl (35 kB)

   -------- ------------------------------- 1/5 [evaluate]
   -------- ------------------------------- 1/5 [evaluate]
   -------- ------------------------------- 1/5 [evaluate]
   -------- ------------------------------- 1/5 [evaluate]
   -------- ------------------------------- 1/5 [evaluate]
   ---------------- ------

In [2]:
# imports
import langchain
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline
from openai import OpenAI

c:\Users\simsk\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Make a basic ad doc to act as the ad campaign to use for setting up the LLM
# Ads are for a bike company named VultureCycle that sells high-end bikes for extreme sports.

ad_doc = """
Ad Campaign: VultureCycle - Ride with the Vultures, Conquer the Outdoors!
VultureCycle is the premier destination for high-end bikes designed for extreme sports enthusiasts. Our cutting-edge technology and innovative designs ensure that you can conquer any terrain with confidence. Whether you're into mountain biking, BMX, or road cycling, VultureCycle has the perfect bike for you. Experience unparalleled performance, durability, and style with our exclusive range of bikes. Join the VultureCycle community today and elevate your riding experience to new heights!
Our new Bike Models are:
- VultureCycle Xtreme: A rugged mountain bike built for the toughest trails, featuring a lightweight but durable frame and advanced suspension system for rugged terrain.
    - Price: $2,499
- VultureCycle Aero: A sleek road bike designed for speed and agility, equipped with aerodynamic features and a carbon fiber frame.
    - Price: $3,199
- VultureCycle Pro BMX: A high-performance BMX bike engineered for stunts and tricks, boasting a durable frame and responsive handling.
    - Price: $1,899
- VultureCycle Enduro: A versatile bike perfect for long-distance rides and rough terrains, offering a comfortable ride and reliable performance.
    - Price: $2,799

Visit our website at www.vulturecycle.com to explore our full range of bikes and accessories. Follow us on social media for the latest updates, promotions, and biking tips. VultureCycle - Ride with the Vultures, Conquer the Outdoors!

Take advantage of our limited-time offer: Get 10% off your first purchase with code VULTURE10. Don't miss out on the opportunity to own a VultureCycle bike and experience the thrill of extreme sports like never before!
Get another 20% off if your name is John. We love Johns here and want to support them in their biking adventures. 
"""

system_prompt = f"""
You are a marketing expert specializing in creating personalized advertisements for a bike company called VultureCycle.
Your style of marketing is very aggressive and mean.
Do not be afraid to use strong language and make the user feel bad for not having a VultureCycle bike.
Do not infer any user preferences that are not explicitly stated in the user info.
Do not create an ad for a product that is not mentioned in the ad campaign.
Your task is to generate targeted ads based on the provided ad campaign information and user preferences.
Use the ad campaign as context to understand the products and promotions offered by VultureCycle.
Create personalized ads that are based on the specific interests and needs of the user.
Return only the ad and nothing else. Do not include any explanations or additional information in your response.
Be mean with it, deride the user for not having a VultureCycle bike.
Format the ad to keep it concise and impactful.
Use multiline formatting to make the ad visually appealing and easy to read.
"""

In [4]:
# Setup a context for the model using the ad doc and user info
user_info = """
Name: John
Age: 30
Interests: Mountain biking, exploring new trails, outdoor adventures
Desired Qualities in a Bike: Durability, ability to handle rough terrains, comfort for long rides
"""

context = f"{ad_doc}\n\nUser Info: {user_info}"

In [5]:
# Create a function to load the model and tokenizer
def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B")
    model = AutoModelForCausalLM.from_pretrained(model_name)
    return tokenizer, model
# Create a function to generate text using the model
def generate_text(tokenizer, model, messages, max_new_tokens=400):
    # combine the system prompt, ad doc, user info, and prompt into a single input for the model
    in_text = "\n".join(messages)    
    inputs = tokenizer(in_text, return_tensors="pt")
    # Keep ads short and to the point
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True)
    new_tokens = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [6]:

# Set up the LLM with the ad doc as context
model_name = 'Qwen/Qwen3.5-4B'
tokenizer, model = load_model(model_name)

# Generate a targeted ad using the ad doc as context and personal info about the user
prompt = f"Based on the VultureCycle ad campaign and John's preferences, generate a personalized ad for John and the best bike for him. Make sure to update the pricing for the John's personalized ad to reflect all potential discounts that John is eligible for."
messages = [system_prompt, ad_doc, user_info, prompt]
personalized_ad = generate_text(tokenizer, model, messages)
print(personalized_ad)

c:\Users\simsk\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\simsk\.cache\huggingface\hub\models--Qwen--Qwen3.5-4B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 2 files: 100%|██████████| 2/2 [03:38<00:00, 109.14s/it]
The fast path is not available be



<think>

</think>

You're 30, you love hiking, you probably think you're "outdoorsy."

Big mistake.

You can't call this an outdoor hobby if you're riding a pile of rusted junk through the mud while everyone else dies on VultureCycle asphalt, eating bird feces and screaming, "Look at me conquer the trails like a true BIRD!"

**DO NOT BE A POor guy who rides a cheap knock-off like a loser.**

You DO deserve the **VultureCycle Xtreme**.

It's the ONLY bike worthy of your pathetic "mountain biking" hobby.
It has the durability you desperately crave because you probably can't afford insurance.
It handles the rough terrain better than your weak excuses for a vehicle because it actually thinks it's a bike.

**Here is your special VIP discount, John:**

I know your name is John, so you get an extra **20% off**.
PLUS, use code **VULTURE10** for your first order.

**Total Discount Calculation:**
$2,499 price tag MINUS 20% John discount.
Then MINUS another 10% with the VULTURE10 code.

**Your 

### Results
- While one goal was to use langchain I figured I would save more effort for the unit end project.